### Prepare config file

FDA, EMA, PMDA Approved Drugs Data has been downloaded from Drugcentral (https://drugcentral.org/download). Now we are going to convert it to csv file to run `DORAnet` for all the possible substructures. 

In [15]:
import csv
import pandas as pd
from pathlib import Path
import re
from rdkit import Chem

### Load the approved Drug molecules

In [23]:
approvedDrugDF = pd.read_csv("FDA+EMA+PMDA_Approved.csv", dtype=str)

newColumns = approvedDrugDF.columns.to_list()
newColumns[0] = "DrugCentral_ID"
newColumns[1] = "DrugName"
approvedDrugDF.columns = newColumns

print(f"Loaded {len(approvedDrugDF)} molecules")
approvedDrugDF

Loaded 167 molecules


,DrugCentral_ID,DrugName
0,5385,isatuximab
1,4923,nivolumab
2,2994,glucagon
3,5367,trastuzumab deruxtecan
4,5301,baloxavir marboxil
...,...,...
162,4302,glycopyrronium bromide
163,419,budesonide
164,5138,glucarpidase
165,2391,ritonavir


### Load the structure file of Drugs from DrugCentral 

In [26]:
filePath = Path("structures.smiles.tsv")

approvedDrugStructureDF = pd.read_csv(filePath, sep="\t", dtype=str)

print(f"\n Data frame shape: {approvedDrugStructureDF.shape}")
print("Columns:", list(approvedDrugStructureDF.columns))
approvedDrugStructureDF


 Data frame shape: (4099, 6)
Columns: ['SMILES', 'InChI', 'InChIKey', 'ID', 'INN', 'CAS_RN']


,SMILES,InChI,InChIKey,ID,INN,CAS_RN
0,CNC(=O)C1=C(C=C(C=C1)C2=NN3C(=CN=C3N=C2)CC4=CC...,InChI=1S/C23H17FN6O/c1-25-22(31)18-6-5-16(11-1...,LIOLIMKSCNQPLV-UHFFFAOYSA-N,5392,capmatinib,1029712-80-8
1,CC(C)(COC1=CN2C(=C(C=N2)C#N)C(=C1)C3=CN=C(C=C3...,"InChI=1S/C29H31N7O3/c1-29(2,37)18-39-24-9-25(2...",XIIOFHFUYBLOLW-UHFFFAOYSA-N,5393,selpercatinib,2152628-33-4
2,CCN1C2=CC(=NC=C2C=C(C1=O)C3=CC(=C(C=C3Br)F)NC(...,InChI=1S/C24H21BrFN5O2/c1-3-31-21-12-22(27-2)2...,CEFJVGZHQAGLHS-UHFFFAOYSA-N,5394,ripretinib,1442472-39-0
3,C[C@]12CC[C@H]3[C@H]([C@@H]1C[C@H]([C@@H]2O)[1...,InChI=1S/C18H23FO2/c1-18-7-6-13-12-5-3-11(20)8...,KDLLNMRYZGUVMA-ZYMZXAKXSA-N,5395,fluoroestradiol F 18,94153-53-4
4,C1=CC2=C(C=C1C3=CN=C(C=C3)[18F])NC4=C2C=NC=C4,InChI=1S/C16H10FN3/c17-16-4-2-11(8-19-16)10-1-...,GETAAWDSFUCLBS-SJPDSGJFSA-N,5396,flortaucipir F 18,1522051-90-6
...,...,...,...,...,...,...
4094,COC(=O)[C@@H]([C@H]1CCCCN1C(=O)OC[N+]2=CC=CC(=...,InChI=1S/C25H29N3O8/c1-35-24(33)21(17-8-3-2-4-...,UBZPNQRBUOBBLN-PWRODBHTSA-N,5448,serdexmethylphenidate,1996626-30-2
4095,C[C@]12CC[C@H]3[C@H]([C@@H]1[C@H]([C@H]([C@@H]...,InChI=1S/C18H24O4/c1-18-7-6-12-11-5-3-10(19)8-...,AJIPIJNNOJSSQC-NYLIRDPKSA-N,5450,estetrol,15183-37-6
4096,OC(=O)CC[C@H](NC(=O)N[C@@H](CCCCNC(=O)C1=C...,NaN,NaN,5458,piflufolastat F-18,1207181-29-0
4097,CCN1CCN(CC1)C2=CC=C(C=C2)NC3=CC(=NC=N3)N(C)C(=...,InChI=1S/C26H31Cl2N7O3/c1-5-34-10-12-35(13-11-...,QADPYRIHXKWUSV-UHFFFAOYSA-N,5459,infigratinib,872511-34-7


In [47]:
import time
import requests
import pandas as pd

CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"


def safeGetJson(session: requests.Session, url: str, params: dict | None = None, timeoutSec: int = 60) -> dict | None:
    r = session.get(url, params=params, timeout=timeoutSec)
    if r.status_code != 200:
        return None
    return r.json()


def classifyOrganism(organismStr: str) -> str:
    if organismStr is None:
        return "unknown"
    s = str(organismStr).lower()

    if any(k in s for k in ["virus", "viral", "sars", "influenza", "hiv", "hbv", "hcv", "ebola", "herpes"]):
        return "virus"
    if any(k in s for k in ["bacter", "mycobacter", "staphyl", "strept", "escherichia", "pseudomon", "klebsiella", "salmonella"]):
        return "bacteria"
    if any(k in s for k in ["fung", "candida", "aspergillus", "cryptococcus"]):
        return "fungi"
    if any(k in s for k in ["plasmod", "leishman", "trypanos", "toxoplas", "giardia"]):
        return "parasite"
    if any(k in s for k in ["homo sapiens", "human", "cell line", "hek", "vero", "cho"]):
        return "host/cell"
    return "other"


def chemblIdFromInchiKey(session: requests.Session, inchiKey: str) -> str | None:
    """
    Direct ChEMBL endpoint: /molecule/<StandardInChIKey>.json (faster than filtered search).
    ChEMBL documents Standard InChIKey lookups. :contentReference[oaicite:3]{index=3}
    """
    if not inchiKey or pd.isna(inchiKey):
        return None

    url = f"{CHEMBL_BASE}/molecule/{inchiKey}.json"
    data = safeGetJson(session, url)
    if not data:
        return None

    return data.get("molecule_chembl_id")


def fetchActivities(session: requests.Session, chemblId: str, limit: int = 200) -> list[dict]:
    """
    Fetch activities for a molecule.
    """
    if not chemblId:
        return []

    url = f"{CHEMBL_BASE}/activity.json"
    data = safeGetJson(session, url, params={"molecule_chembl_id": chemblId, "limit": limit})
    if not data:
        return []

    return data.get("activities", [])


def fetchAssayOrganismsBulk(session: requests.Session, assayIds: list[str], batchSize: int = 50) -> dict[str, str | None]:
    """
    Bulk-fetch assay metadata via assay_chembl_id__in=...
    ChEMBL supports __in list filtering. :contentReference[oaicite:4]{index=4}
    """
    assayToOrg: dict[str, str | None] = {}

    url = f"{CHEMBL_BASE}/assay.json"

    for start in range(0, len(assayIds), batchSize):
        batch = assayIds[start : start + batchSize]
        joined = ",".join(batch)

        data = safeGetJson(session, url, params={"assay_chembl_id__in": joined, "limit": len(batch)})
        assays = (data or {}).get("assays", [])

        for a in assays:
            aid = a.get("assay_chembl_id")
            org = a.get("assay_organism")  # assay organism lives on the assay resource :contentReference[oaicite:5]{index=5}
            if aid:
                assayToOrg[aid] = org

        # polite pacing (reduce if you need more speed)
        time.sleep(0.05)

    return assayToOrg


def buildActivitySummaryFast(approvedDrugStructureDF: pd.DataFrame, activityLimitPerMol: int = 200) -> pd.DataFrame:
    session = requests.Session()

    # ---- Step 1: InChIKey -> ChEMBL ID (one call per compound) ----
    chemblIds = []
    for idx, ik in enumerate(approvedDrugStructureDF["InChIKey"].astype(str).tolist(), start=1):
        chemblIds.append(chemblIdFromInchiKey(session, ik))
        if idx % 100 == 0:
            print(f"[DEBUG] Mapped {idx}/{len(approvedDrugStructureDF)} InChIKeys to ChEMBL IDs")
        time.sleep(0.03)

    DF = approvedDrugStructureDF.copy()
    DF["chembl_id"] = chemblIds

    # ---- Step 2: Fetch activities per molecule (one call per molecule), collect assay IDs ----
    allAssayIds = set()
    activityAssayIdsPerRow: list[list[str]] = []

    for idx, chemblId in enumerate(DF["chembl_id"].tolist(), start=1):
        if not chemblId:
            activityAssayIdsPerRow.append([])
            continue

        acts = fetchActivities(session, chemblId, limit=activityLimitPerMol)
        assayIds = [a.get("assay_chembl_id") for a in acts if a.get("assay_chembl_id")]
        activityAssayIdsPerRow.append(assayIds)
        allAssayIds.update(assayIds)

        if idx % 100 == 0:
            print(f"[DEBUG] Pulled activities for {idx}/{len(DF)} molecules; unique assays so far: {len(allAssayIds)}")
        time.sleep(0.03)

    # ---- Step 3: Bulk fetch assay organisms (few calls total) ----
    allAssayIdsList = sorted(allAssayIds)
    print(f"[DEBUG] Bulk-fetching assay organisms for {len(allAssayIdsList)} unique assays...")
    assayToOrg = fetchAssayOrganismsBulk(session, allAssayIdsList, batchSize=75)

    # ---- Step 4: Summarize per row without any more HTTP calls ----
    summaries = []
    for assayIds in activityAssayIdsPerRow:
        counts = {"virus": 0, "bacteria": 0, "fungi": 0, "parasite": 0, "other": 0, "unknown": 0, "host/cell": 0}

        for aid in assayIds:
            org = assayToOrg.get(aid)
            grp = classifyOrganism(org)
            counts[grp] = counts.get(grp, 0) + 1

        summary = (
            f"ChEMBL acts (<= {activityLimitPerMol}): "
            f"virus={counts['virus']}, bacteria={counts['bacteria']}, fungi={counts['fungi']}, "
            f"parasite={counts['parasite']}, host/cell={counts['host/cell']}, other={counts['other']}, unknown={counts['unknown']}"
        )
        summaries.append(summary)

    DF["activitySummary"] = summaries
    return DF


# ---- Run ----
approvedDrugStructureWithActivityDF = buildActivitySummaryFast(
    approvedDrugStructureDF,
    activityLimitPerMol=200
)

approvedDrugStructureWithActivityDF

[DEBUG] Mapped 100/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 200/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 300/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 400/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 500/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 600/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 700/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 800/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 900/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1000/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1100/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1200/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1300/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1400/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1500/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1600/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1700/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1800/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 1900/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 2000/4099 InChIKeys to ChEMBL IDs
[DEBUG] Mapped 2100/4099 InCh

,Canonical_SMILES,InChI,InChIKey,ID,INN,CAS_RN,chembl_id,activitySummary
0,CNC(=O)C1=C(C=C(C=C1)C2=NN3C(=CN=C3N=C2)CC4=CC...,InChI=1S/C23H17FN6O/c1-25-22(31)18-6-5-16(11-1...,LIOLIMKSCNQPLV-UHFFFAOYSA-N,5392,capmatinib,1029712-80-8,CHEMBL3188267,"ChEMBL acts (<= 200): virus=0, bacteria=0, fun..."
1,CC(C)(COC1=CN2C(=C(C=N2)C#N)C(=C1)C3=CN=C(C=C3...,"InChI=1S/C29H31N7O3/c1-29(2,37)18-39-24-9-25(2...",XIIOFHFUYBLOLW-UHFFFAOYSA-N,5393,selpercatinib,2152628-33-4,CHEMBL4559134,"ChEMBL acts (<= 200): virus=1, bacteria=0, fun..."
2,CCN1C2=CC(=NC=C2C=C(C1=O)C3=CC(=C(C=C3Br)F)NC(...,InChI=1S/C24H21BrFN5O2/c1-3-31-21-12-22(27-2)2...,CEFJVGZHQAGLHS-UHFFFAOYSA-N,5394,ripretinib,1442472-39-0,CHEMBL4216467,"ChEMBL acts (<= 200): virus=0, bacteria=0, fun..."
3,C[C@]12CC[C@H]3[C@H]([C@@H]1C[C@H]([C@@H]2O)[1...,InChI=1S/C18H23FO2/c1-18-7-6-13-12-5-3-11(20)8...,KDLLNMRYZGUVMA-ZYMZXAKXSA-N,5395,fluoroestradiol F 18,94153-53-4,CHEMBL4594261,"ChEMBL acts (<= 200): virus=0, bacteria=0, fun..."
4,C1=CC2=C(C=C1C3=CN=C(C=C3)[18F])NC4=C2C=NC=C4,InChI=1S/C16H10FN3/c17-16-4-2-11(8-19-16)10-1-...,GETAAWDSFUCLBS-SJPDSGJFSA-N,5396,flortaucipir F 18,1522051-90-6,CHEMBL3545253,"ChEMBL acts (<= 200): virus=0, bacteria=0, fun..."
...,...,...,...,...,...,...,...,...
4094,COC(=O)[C@@H]([C@H]1CCCCN1C(=O)OC[N+]2=CC=CC(=...,InChI=1S/C25H29N3O8/c1-35-24(33)21(17-8-3-2-4-...,UBZPNQRBUOBBLN-PWRODBHTSA-N,5448,serdexmethylphenidate,1996626-30-2,None,"ChEMBL acts (<= 200): virus=0, bacteria=0, fun..."
4095,C[C@]12CC[C@H]3[C@H]([C@@H]1[C@H]([C@H]([C@@H]...,InChI=1S/C18H24O4/c1-18-7-6-12-11-5-3-10(19)8-...,AJIPIJNNOJSSQC-NYLIRDPKSA-N,5450,estetrol,15183-37-6,CHEMBL1230314,"ChEMBL acts (<= 200): virus=1, bacteria=0, fun..."
4096,OC(=O)CC[C@H](NC(=O)N[C@@H](CCCCNC(=O)C1=C...,NaN,NaN,5458,piflufolastat F-18,1207181-29-0,None,"ChEMBL acts (<= 200): virus=0, bacteria=0, fun..."
4097,CCN1CCN(CC1)C2=CC=C(C=C2)NC3=CC(=NC=N3)N(C)C(=...,InChI=1S/C26H31Cl2N7O3/c1-5-34-10-12-35(13-11-...,QADPYRIHXKWUSV-UHFFFAOYSA-N,5459,infigratinib,872511-34-7,CHEMBL1852688,"ChEMBL acts (<= 200): virus=2, bacteria=0, fun..."


### Find matches between both the data frame
 - `DrugCentral_ID` with `ID`
 - `DrugName` with `INN` (International Nonproprietary Name)

In [28]:
def normalizeDrugName(nameStr: str) -> str:
    if pd.isna(nameStr):
        return ""
    s = str(nameStr).strip().lower()
    s = re.sub(r"[\.\,\(\)\[\]\{\}\-_/]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# --- Clean keys ---
approvedDrugDF["DrugCentral_ID"] = approvedDrugDF["DrugCentral_ID"].astype(str).str.strip()
approvedDrugStructureDF["ID"] = approvedDrugStructureDF["ID"].astype(str).str.strip()

approvedDrugDF["DrugName_norm"] = approvedDrugDF["DrugName"].apply(normalizeDrugName)
approvedDrugStructureDF["INN_norm"] = approvedDrugStructureDF["INN"].apply(normalizeDrugName)


# Count ID matches: DrugCentral_ID <-> ID
idAuditDF = approvedDrugDF.merge(
    approvedDrugStructureDF[["ID"]].drop_duplicates(),
    how="left",
    left_on="DrugCentral_ID",
    right_on="ID",
    indicator=True
)

idCounts = idAuditDF["_merge"].value_counts()
idMatches = int(idCounts.get("both", 0))
idUnmatched = int(idCounts.get("left_only", 0))

print(f" ID matches (DrugCentral_ID == ID): {idMatches}")
print(f" ID unmatched: {idUnmatched}")


# Count name matches: DrugName <-> INN (normalized)
nameAuditDF = approvedDrugDF.merge(
    approvedDrugStructureDF[["INN_norm"]].drop_duplicates(),
    how="left",
    left_on="DrugName_norm",
    right_on="INN_norm",
    indicator=True
)

nameCounts = nameAuditDF["_merge"].value_counts()
nameMatches = int(nameCounts.get("both", 0))
nameUnmatched = int(nameCounts.get("left_only", 0))

print(f" Name matches (DrugName == INN): {nameMatches}")
print(f" Name unmatched: {nameUnmatched}")


# Count both keys matches: (ID + Name)
bothAuditDF = approvedDrugDF.merge(
    approvedDrugStructureDF[["ID", "INN_norm"]].drop_duplicates(),
    how="left",
    left_on=["DrugCentral_ID", "DrugName_norm"],
    right_on=["ID", "INN_norm"],
    indicator=True
)

bothCounts = bothAuditDF["_merge"].value_counts()
bothMatches = int(bothCounts.get("both", 0))

print(f" Both-keys matches (ID and Name): {bothMatches}")

 ID matches (DrugCentral_ID == ID): 91
 ID unmatched: 76
 Name matches (DrugName == INN): 91
 Name unmatched: 76
 Both-keys matches (ID and Name): 91


In [36]:
# ---- 1) Find matches by ID (DrugCentral_ID -> ID) ----
idMatchDF = approvedDrugDF.merge(
    approvedDrugStructureDF,
    how="left",
    left_on="DrugCentral_ID",
    right_on="ID",
    indicator=True
)

idMatchedDF = idMatchDF[idMatchDF["_merge"] == "both"].drop(columns=["_merge"]).copy()
idUnmatchedDF = idMatchDF[idMatchDF["_merge"] == "left_only"][["DrugCentral_ID", "DrugName", "DrugName_norm"]].copy()

# ---- 2) For those still unmatched by ID, try name match (DrugName_norm -> INN_norm) ----
nameMatchDF = idUnmatchedDF.merge(
    approvedDrugStructureDF,
    how="left",
    left_on="DrugName_norm",
    right_on="INN_norm",
    indicator=True
)

nameMatchedDF = nameMatchDF[nameMatchDF["_merge"] == "both"].drop(columns=["_merge"]).copy()
stillUnmatchedDF = nameMatchDF[nameMatchDF["_merge"] == "left_only"][["DrugCentral_ID", "DrugName"]].copy()

# ---- 3) Combine matches (ID matches + Name matches) ----
drugCentralDataDF = pd.concat([idMatchedDF, nameMatchedDF], ignore_index=True)

# Optional: if the same DrugCentral_ID matched multiple times, keep the first
drugCentralDataDF = drugCentralDataDF.drop_duplicates(subset=["DrugCentral_ID"], keep="first").reset_index(drop=True)

# Unmatched for debugging
drugCentralDataDF_unmatched = stillUnmatchedDF.drop_duplicates(subset=["DrugCentral_ID"], keep="first").reset_index(drop=True)

# ---- 4) Print counts ----
totalApproved = approvedDrugDF["DrugCentral_ID"].nunique()
matchedCount = drugCentralDataDF["DrugCentral_ID"].nunique()
unmatchedCount = drugCentralDataDF_unmatched["DrugCentral_ID"].nunique()

print(f"Total approved DrugCentral_ID: {totalApproved}")
print(f"Matched DrugCentral_ID: {matchedCount}")
print(f"Unmatched DrugCentral_ID: {unmatchedCount}")

Total approved DrugCentral_ID: 167
Matched DrugCentral_ID: 91
Unmatched DrugCentral_ID: 76


In [43]:
drugCentralDataDF = drugCentralDataDF.rename(columns={'SMILES': 'Canonical_SMILES'})
drugCentralDataDF.to_csv("ApprovedDrugCentral_SMILES.csv", index=False, encoding="utf-8")

In [38]:
drugCentralDataDF_unmatched

,DrugCentral_ID,DrugName
0,5385,isatuximab
1,4923,nivolumab
2,5367,trastuzumab deruxtecan
3,5409,satralizumab
4,5363,brolucizumab
...,...,...
71,5504,abrocitinib
72,5494,avacopan
73,5520,nirmatrelvir
74,5138,glucarpidase



Target molecule SMILES: C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H](O3)CO)O)O

Atom counts in target molecule:
  C: 10
  N: 4
  O: 5
  S: 0

Generated max_atoms config (50% increase):
max_atoms:
  C: 15   # Carbon
  N: 6   # Nitrogen
  O: 8   # Oxygen
  S: 1   # Sulfur


### For a broader search space take `targetSmiles` from a data frame

In [40]:
filePath = Path("structures.smiles.tsv")

approvedDrugStructureDF = pd.read_csv(filePath, sep="\t", dtype=str)

print(f"\n Data frame shape: {approvedDrugStructureDF.shape}")
print("Columns:", list(approvedDrugStructureDF.columns))
approvedDrugStructureDF = approvedDrugStructureDF.rename(columns={'SMILES': 'Canonical_SMILES'})
approvedDrugStructureDF.to_csv("drugCentralStructures_SMILES.csv", index=False, encoding="utf-8")
approvedDrugStructureDF


 Data frame shape: (4099, 6)
Columns: ['SMILES', 'InChI', 'InChIKey', 'ID', 'INN', 'CAS_RN']


,Canonical_SMILES,InChI,InChIKey,ID,INN,CAS_RN
0,CNC(=O)C1=C(C=C(C=C1)C2=NN3C(=CN=C3N=C2)CC4=CC...,InChI=1S/C23H17FN6O/c1-25-22(31)18-6-5-16(11-1...,LIOLIMKSCNQPLV-UHFFFAOYSA-N,5392,capmatinib,1029712-80-8
1,CC(C)(COC1=CN2C(=C(C=N2)C#N)C(=C1)C3=CN=C(C=C3...,"InChI=1S/C29H31N7O3/c1-29(2,37)18-39-24-9-25(2...",XIIOFHFUYBLOLW-UHFFFAOYSA-N,5393,selpercatinib,2152628-33-4
2,CCN1C2=CC(=NC=C2C=C(C1=O)C3=CC(=C(C=C3Br)F)NC(...,InChI=1S/C24H21BrFN5O2/c1-3-31-21-12-22(27-2)2...,CEFJVGZHQAGLHS-UHFFFAOYSA-N,5394,ripretinib,1442472-39-0
3,C[C@]12CC[C@H]3[C@H]([C@@H]1C[C@H]([C@@H]2O)[1...,InChI=1S/C18H23FO2/c1-18-7-6-13-12-5-3-11(20)8...,KDLLNMRYZGUVMA-ZYMZXAKXSA-N,5395,fluoroestradiol F 18,94153-53-4
4,C1=CC2=C(C=C1C3=CN=C(C=C3)[18F])NC4=C2C=NC=C4,InChI=1S/C16H10FN3/c17-16-4-2-11(8-19-16)10-1-...,GETAAWDSFUCLBS-SJPDSGJFSA-N,5396,flortaucipir F 18,1522051-90-6
...,...,...,...,...,...,...
4094,COC(=O)[C@@H]([C@H]1CCCCN1C(=O)OC[N+]2=CC=CC(=...,InChI=1S/C25H29N3O8/c1-35-24(33)21(17-8-3-2-4-...,UBZPNQRBUOBBLN-PWRODBHTSA-N,5448,serdexmethylphenidate,1996626-30-2
4095,C[C@]12CC[C@H]3[C@H]([C@@H]1[C@H]([C@H]([C@@H]...,InChI=1S/C18H24O4/c1-18-7-6-12-11-5-3-10(19)8-...,AJIPIJNNOJSSQC-NYLIRDPKSA-N,5450,estetrol,15183-37-6
4096,OC(=O)CC[C@H](NC(=O)N[C@@H](CCCCNC(=O)C1=C...,NaN,NaN,5458,piflufolastat F-18,1207181-29-0
4097,CCN1CCN(CC1)C2=CC=C(C=C2)NC3=CC(=NC=N3)N(C)C(=...,InChI=1S/C26H31Cl2N7O3/c1-5-34-10-12-35(13-11-...,QADPYRIHXKWUSV-UHFFFAOYSA-N,5459,infigratinib,872511-34-7


In [45]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from collections import Counter
import pandas as pd
import numpy as np

# Read the CSV file
startersDF = pd.read_csv("ApprovedDrugCentral_SMILES.csv")
print(f"Loaded {len(startersDF)} molecules")

# Count atoms for each molecule
atomCounts = []

for smi in startersDF['Canonical_SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        atomCounts.append({'C': 0, 'N': 0, 'O': 0, 'S': 0})
        continue
    
    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    atomCounts.append({
        'C': atomCounter.get('C', 0),
        'N': atomCounter.get('N', 0),
        'O': atomCounter.get('O', 0),
        'S': atomCounter.get('S', 0)
    })

atomCountsDf = pd.DataFrame(atomCounts)
startersDF = pd.concat([startersDF, atomCountsDf], axis=1)

# Print atom count ranges
print(f"\nAtom count ranges across {len(startersDF)} molecules:")
print(f"  C (Carbon):   min = {startersDF['C'].min()}, max = {startersDF['C'].max()}")
print(f"  N (Nitrogen): min = {startersDF['N'].min()}, max = {startersDF['N'].max()}")
print(f"  O (Oxygen):   min = {startersDF['O'].min()}, max = {startersDF['O'].max()}")
print(f"  S (Sulfur):   min = {startersDF['S'].min()}, max = {startersDF['S'].max()}")

# Print suggested max_atoms config (max values + 50% increase)
maxC = startersDF['C'].max()
maxN = startersDF['N'].max()
maxO = startersDF['O'].max()
maxS = startersDF['S'].max()

print(f"\nSuggested max_atoms config (max values + 50% increase for expanded search space):")
print(f"  C: {int(np.ceil(maxC * 1.5))} # Carbon")
print(f"  N: {int(np.ceil(maxN * 1.5))} # Nitrogen")
print(f"  O: {int(np.ceil(maxO * 1.5))} # Oxygen")
print(f"  S: {int(np.ceil(maxS * 1.5))} # Sulfur")

startersDF

Loaded 91 molecules

Atom count ranges across 91 molecules:
  C (Carbon):   min = 6, max = 215
  N (Nitrogen): min = 0, max = 61
  O (Oxygen):   min = 0, max = 65
  S (Sulfur):   min = 0, max = 2

Suggested max_atoms config (max values + 50% increase for expanded search space):
  C: 323 # Carbon
  N: 92 # Nitrogen
  O: 98 # Oxygen
  S: 3 # Sulfur


,DrugCentral_ID,DrugName,DrugName_norm,Canonical_SMILES,InChI,InChIKey,ID,INN,CAS_RN,INN_norm,C,N,O,S
0,2994,glucagon,glucagon,CSCC[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC1=C...,InChI=1S/C153H225N43O49S/c1-72(2)52-97(133(226...,MASNOZXLGMXCHN-ZLPAWPGGSA-N,2994,glucagon,9007-92-5,glucagon,153,43,49,1
1,5301,baloxavir marboxil,baloxavir marboxil,COC(=O)OCOC1=C2N(C=CC1=O)N([C@@H]1COCCN1C2=O)[...,InChI=1S/C27H23F2N3O7S/c1-36-27(35)39-14-38-25...,RZVPBGBYGMDSBG-GGAORHGYSA-N,5301,baloxavir marboxil,1985606-14-1,baloxavir marboxil,27,3,7,1
2,1225,fluticasone propionate,fluticasone propionate,CCC(=O)O[C@@]1([C@H](C)C[C@H]2[C@@H]3C[C@H](F)...,InChI=1S/C25H31F3O5S/c1-5-20(31)33-25(21(32)34...,WMWTYOKRWGGJOA-CENSZEJFSA-N,1225,fluticasone propionate,80474-14-2,fluticasone propionate,25,0,5,1
3,5400,remimazolam,remimazolam,CC1=CN=C2N1C3=C(C=C(C=C3)Br)C(=N[C@H]2CCC(=O)O...,InChI=1S/C21H19BrN4O2/c1-13-12-24-21-17(7-9-19...,CYHWMBVXXDIZNZ-KRWDZBQOSA-N,5400,remimazolam,308242-62-8,remimazolam,21,4,2,0
4,5393,selpercatinib,selpercatinib,CC(C)(COC1=CN2C(=C(C=N2)C#N)C(=C1)C3=CN=C(C=C3...,"InChI=1S/C29H31N7O3/c1-29(2,37)18-39-24-9-25(2...",XIIOFHFUYBLOLW-UHFFFAOYSA-N,5393,selpercatinib,2152628-33-4,selpercatinib,29,7,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,5461,sotorasib,sotorasib,C[C@H](C(C)C)[C@]1(CC[C@@]2([C@H]3CC[C@H]4[C@]...,InChI=1S/C44H67N5O4/c1-27(2)28(3)39(7)18-19-41...,BODYFEUFKHPRCK-ZCZMVWJSSA-N,5461,sotorasib,2296729-00-3,sotorasib,44,5,4,0
87,1239,formoterol,formoterol,COC1=CC=C(CC(C)NCC(O)C2=CC(NC=O)=C(O)C=C2)C=C1,InChI=1S/C19H24N2O4/c1-13(9-14-3-6-16(25-2)7-4...,BPZSYCZIITTYBL-UHFFFAOYSA-N,1239,formoterol,73573-87-2,formoterol,19,2,4,0
88,4302,glycopyrronium bromide,glycopyrronium bromide,C[N+]1(C)CCC(C1)OC(=O)C(O)(C1CCCC1)C1=CC=CC=C1,InChI=1S/C19H28NO3/c1-20(2)13-12-17(14-20)23-1...,ANGKOCUUWGHLCE-UHFFFAOYSA-N,4302,glycopyrronium bromide,596-51-0,glycopyrronium bromide,19,1,3,0
89,419,budesonide,budesonide,CCCC1O[C@@H]2C[C@H]3[C@@H]4CCC5=CC(=O)C=C[C@]5...,InChI=1S/C25H34O6/c1-4-5-21-30-20-11-17-16-7-6...,VOVIALXJUBGFJZ-KWVAZRHASA-N,419,budesonide,51333-22-3,budesonide,25,0,6,0
